In [6]:
# -*- coding: utf-8 -*-
r"""
Metadata-only fetcher for GitHub repos in URL_List.csv (no sampling)

- Reads GitHub URLs from: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Manual\URL_List.csv
- Uses tokens from All_tokens.env (GITHUB_TOKEN_1..6)
- Fetches repo metadata + paginated counts: contributors, pull requests, commits
- Safely appends rows to Project_Metadata.csv with Windows-friendly retry; falls back to *.pending.<ts>.csv if locked
- On each run, auto-merges any sidecar files back into Project_Metadata.csv once it is writable
"""

from __future__ import annotations
import os, time, stat, glob
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse

import pandas as pd
import requests
from dotenv import load_dotenv

# ========= CONFIG =========
ENV_FILE = 'All_tokens.env'
MAX_PROJECTS = 100_000          # hard upper bound to avoid accidents
START_NUMBER = 1                # 1-based start row in URL_List.csv

# === LOAD .env ===
load_dotenv(ENV_FILE)

TOKENS = [os.getenv(f'GITHUB_TOKEN_{i}') for i in range(1, 7)]
TOKENS = [t for t in TOKENS if t]
if not TOKENS:
    print("⚠️ Warning: No GitHub tokens found in All_tokens.env (API calls may be rate-limited).")
token_index = 0  # for rotation

# === PATHS ===
base_dir = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Manual")
csv_path = base_dir / "URL_List.csv"
metadata_path = base_dir / "Project_Metadata.csv"
base_dir.mkdir(parents=True, exist_ok=True)

# ========= SAFE APPEND (Windows-friendly) =========
def _ensure_dir_writable(p: Path):
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists():
        try:
            os.chmod(p, stat.S_IWRITE | stat.S_IREAD)
        except Exception:
            pass
    try:
        os.chmod(p.parent, stat.S_IWRITE | stat.S_IREAD | stat.S_IEXEC)
    except Exception:
        pass

def _needs_header(csv_path: Path) -> bool:
    """Return True if the file is missing or zero-length."""
    return (not csv_path.exists()) or (os.path.getsize(csv_path) == 0)

def append_csv_with_retry(csv_path: Path, df: pd.DataFrame, max_tries: int = 8, base_sleep: float = 0.6) -> Path:
    """
    Append rows to CSV with retries on PermissionError (e.g., Excel lock).
    Uses CRLF line endings for Excel friendliness.
    Returns the path actually written to (original or *.pending.<ts>.csv sidecar).
    """
    _ensure_dir_writable(csv_path)
    header = _needs_header(csv_path)
    attempt = 0
    while attempt < max_tries:
        try:
            # If file does not exist, try exclusive create ('x'); else append ('a')
            df.to_csv(
                csv_path,
                mode='a' if csv_path.exists() else 'x',
                header=header,
                index=False,
                encoding='utf-8',
                lineterminator='\r\n'  # CRLF for Excel
            )
            print(f"📝 Wrote {len(df)} row(s) to: {csv_path} (size={os.path.getsize(csv_path)} bytes)")
            return csv_path
        except PermissionError:
            attempt += 1
            time.sleep(base_sleep * attempt)
        except FileExistsError:
            # Race between exists() and 'x' mode; switch to append without header
            header = False
    # Final fallback: write to sidecar to avoid data loss
    sidecar = csv_path.with_suffix(csv_path.suffix + f".pending.{int(time.time())}.csv")
    df.to_csv(sidecar, index=False, encoding='utf-8', lineterminator='\r\n')
    print(f"⚠️ Locked target. Wrote to sidecar: {sidecar} (size={os.path.getsize(sidecar)} bytes)")
    return sidecar

def merge_sidecars_into_metadata(metadata_csv: Path, pattern: str = None):
    """
    Merge any sidecar pending files into the main metadata CSV once it's writable.
    Sidecars are then removed on success.
    """
    _ensure_dir_writable(metadata_csv)
    if pattern is None:
        pattern = metadata_csv.with_suffix(metadata_csv.suffix + ".pending.*.csv").as_posix()
    sidecars = sorted(glob.glob(pattern))
    if not sidecars:
        return

    print(f"🔄 Found {len(sidecars)} pending sidecar(s) to merge.")

    # Read/prepare current header state
    header_needed = _needs_header(metadata_csv)

    for sc in sidecars:
        try:
            df = pd.read_csv(sc)
            # Append with retries
            written_path = append_csv_with_retry(metadata_csv, df)
            if written_path == metadata_csv:
                # On success, delete sidecar
                try:
                    os.remove(sc)
                    print(f"🧹 Removed sidecar after merge: {sc}")
                except Exception as e_rm:
                    print(f"⚠️ Could not remove sidecar {sc}: {e_rm}")
            else:
                # Could not merge back; keep sidecar
                print(f"ℹ️ Sidecar {sc} still present (main file locked).")
        except Exception as e:
            print(f"⚠️ Failed to merge sidecar {sc}: {e}")

# ========= API HELPERS =========
def next_auth_headers() -> dict:
    global token_index
    if TOKENS:
        headers = {'Authorization': f'token {TOKENS[token_index % len(TOKENS)]}'}
        token_index += 1
        return headers
    return {}

def get_json(url: str, headers: Optional[dict] = None, params: Optional[dict] = None, timeout: int = 30) -> dict:
    try:
        r = requests.get(url, headers=headers or {}, params=params or {}, timeout=timeout)
        if r.status_code == 200:
            return r.json()
        # Try again with the next token if rate-limited or unauthorized
        if r.status_code in (401, 403):
            r2 = requests.get(url, headers=next_auth_headers(), params=params or {}, timeout=timeout)
            return r2.json() if r2.status_code == 200 else {}
        return {}
    except Exception:
        return {}

def get_count(api_url: str) -> int:
    """
    Paginate a list endpoint and count items; rotates tokens across pages.
    """
    per_page = 100
    page = 1
    total = 0
    while True:
        headers = next_auth_headers()
        try:
            r = requests.get(api_url, headers=headers, params={"per_page": per_page, "page": page}, timeout=30)
            if r.status_code != 200:
                break
            items = r.json()
            if not isinstance(items, list):
                break
            total += len(items)
            if len(items) < per_page:
                break
            page += 1
        except Exception:
            break
    return total

# ========= LOAD URL LIST =========
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
(df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index'))

total = min(len(df), MAX_PROJECTS)
print(f"🔎 Ready: {total} repos to process (no sampling).")

# Try to merge any leftover sidecars at the start (if the CSV got unlocked since last run)
merge_sidecars_into_metadata(metadata_path)

# ========= PROCESS: METADATA ONLY =========
for idx in range(START_NUMBER - 1, total):
    url = df.iloc[idx]['github_url'].strip()
    owner_repo = urlparse(url).path.strip("/").split("/")
    if len(owner_repo) < 2:
        continue
    owner, project = owner_repo[0], owner_repo[1]
    repo_index = str(idx).zfill(4)
    repo_name = f"{repo_index}.{owner}.{project}"

    print(f"\n🔍 [{idx + 1}/{total}] Processing {repo_name}...")

    base_api = f"https://api.github.com/repos/{owner}/{project}"

    # --- Base repo metadata ---
    data = get_json(base_api, headers=next_auth_headers())

    # --- Counts (paginated) ---
    contributors_count = get_count(f"{base_api}/contributors")
    pulls_count = get_count(f"{base_api}/pulls?state=all")
    commits_count = get_count(f"{base_api}/commits")

    metadata_row = {
        "html_url": url,
        "repo_index": repo_index,
        "repo_name": repo_name,
        "id": data.get("id"),
        "name": data.get("name"),
        "full_name": data.get("full_name"),
        "owner": (data.get("owner") or {}).get("login"),
        "private": data.get("private"),
        "fork": data.get("fork"),
        "created_at": data.get("created_at"),
        "updated_at": data.get("updated_at"),
        "pushed_at": data.get("pushed_at"),
        "homepage": data.get("homepage"),
        "size": data.get("size"),
        "stargazers_count": data.get("stargazers_count"),
        "language": data.get("language"),
        "forks_count": data.get("forks_count"),
        "open_issues_count": data.get("open_issues_count"),
        "license": (data.get("license") or {}).get("name"),
        "topics": ", ".join(data.get("topics", [])) if isinstance(data.get("topics"), list) else None,
        "visibility": data.get("visibility"),
        "default_branch": data.get("default_branch"),
        "has_issues": data.get("has_issues"),
        "has_projects": data.get("has_projects"),
        "has_downloads": data.get("has_downloads"),
        "has_wiki": data.get("has_wiki"),
        "has_pages": data.get("has_pages"),
        "archived": data.get("archived"),
        "disabled": data.get("disabled"),
        "allow_forking": data.get("allow_forking"),
        "is_template": data.get("is_template"),
        "web_commit_signoff_required": data.get("web_commit_signoff_required"),
        "contributors": contributors_count,
        "pull_requests": pulls_count,
        "commits_GitAPI": commits_count
    }

    # --- Safe append to Project_Metadata.csv ---
    md_df = pd.DataFrame([metadata_row])
    written_path = append_csv_with_retry(metadata_path, md_df)
    if written_path != metadata_path:
        # If we wrote to a sidecar, no merge now (likely still locked); will merge next run
        pass

# Attempt a final merge of any sidecars (if unlocked now)
merge_sidecars_into_metadata(metadata_path)

print("\n✅ Process complete. Metadata-only mode with safe CSV appends (no sampling, no dedup, no contributor exports).")


🔎 Ready: 2 repos to process (no sampling).

🔍 [1/2] Processing 0000.Xposed-Modules-Repo.io.github.lsposed.disableflagsecure...
📝 Wrote 1 row(s) to: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Manual\Project_Metadata.csv (size=849 bytes)

🔍 [2/2] Processing 0001.david-legend.david-legend.github.io...
📝 Wrote 1 row(s) to: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Manual\Project_Metadata.csv (size=1338 bytes)

✅ Process complete. Metadata-only mode with safe CSV appends (no sampling, no dedup, no contributor exports).
